# CPSTN Enhanced — PEMS-BAY Training (FIXED)

This notebook is the corrected training pipeline.

**Dataset:** PEMS-BAY, 325 sensors, 5-minute traffic-speed observations.

**Experiment design**
- Chronological split: 70% train / 10% validation / 20% test
- Train-only normalization
- 12 historical steps → 6 future steps
- Official PEMS-BAY adjacency matrix
- Speed + time-of-day sin/cos features
- Gated temporal convolution + graph convolution + residual blocks
- AdamW + ReduceLROnPlateau + early stopping
- Best checkpoint selected **only on validation loss**
- The test set is NOT used during training/model selection

The separate testing notebook performs the final test evaluation.

In [ ]:
# 1. Environment
!nvidia-smi
!pip -q install pandas numpy matplotlib tqdm

import os, time, random, pickle, hashlib, urllib.request, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required. In Colab: Runtime -> Change runtime type -> GPU.")

device = torch.device("cuda")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# 2. Download + verify the exact public CSV distribution
DATA_DIR = Path("/content/pems_bay")
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE = "https://zenodo.org/records/5146275/files"
DATA_URL = f"{BASE}/PEMS-BAY.csv?download=1"
ADJ_URL = f"{BASE}/adj_mx_bay.pkl?download=1"

CSV_PATH = DATA_DIR / "PEMS-BAY.csv"
ADJ_PATH = DATA_DIR / "adj_mx_bay.pkl"

EXPECTED_MD5 = {
    "PEMS-BAY.csv": "c8dea58987a5882e946217c22fdb8256",
    "adj_mx_bay.pkl": "55d25daceac847312b7748c59ded6f77",
}

def md5(path, chunk=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def download_verified(url, path):
    expected = EXPECTED_MD5[path.name]
    if path.exists() and path.stat().st_size > 0 and md5(path) == expected:
        print("Verified existing:", path.name)
        return
    if path.exists():
        path.unlink()
    print("Downloading:", path.name)
    urllib.request.urlretrieve(url, path)
    actual = md5(path)
    if actual != expected:
        raise RuntimeError(f"MD5 mismatch for {path.name}: {actual} != {expected}")
    print("Verified:", path.name, "|", round(path.stat().st_size/1024**2, 2), "MB")

download_verified(DATA_URL, CSV_PATH)
download_verified(ADJ_URL, ADJ_PATH)

In [ ]:
# 3. Load and validate traffic data
df = pd.read_csv(CSV_PATH, index_col=0)
df.index = pd.to_datetime(df.index)

if df.shape[1] != 325:
    raise ValueError(f"Expected 325 sensors, got {df.shape[1]}")

if not df.index.is_monotonic_increasing:
    raise ValueError("Timestamps are not sorted.")

delta = df.index.to_series().diff().dropna()
print("Shape:", df.shape)
print("Time:", df.index.min(), "->", df.index.max())
print("Median sampling interval:", delta.median())

df = df.apply(pd.to_numeric, errors="coerce")
missing_before = int(df.isna().sum().sum())
df = df.interpolate(method="linear", limit_direction="both").ffill().bfill()

if df.isna().any().any():
    raise RuntimeError("NaNs remain after preprocessing.")

speed = df.to_numpy(dtype=np.float32)
timestamps = df.index
print("Missing before:", missing_before)
print("Missing after :", int(np.isnan(speed).sum()))
print("Sensors       :", speed.shape[1])

In [ ]:
# 4. Load the official PEMS-BAY adjacency
with open(ADJ_PATH, "rb") as f:
    obj = pickle.load(f, encoding="latin1")

if isinstance(obj, (tuple, list)) and len(obj) >= 3:
    raw_adj = np.asarray(obj[-1], dtype=np.float32)
else:
    raw_adj = np.asarray(obj, dtype=np.float32)

if raw_adj.shape != (325, 325):
    raise ValueError(f"Expected 325x325 adjacency, got {raw_adj.shape}")

A = np.nan_to_num(raw_adj, nan=0.0, posinf=0.0, neginf=0.0)
A = np.maximum(A, A.T)
np.fill_diagonal(A, 1.0)

degree = A.sum(axis=1)
A_norm = (A / np.sqrt(degree[:, None] * degree[None, :] + 1e-8)).astype(np.float32)
A_tensor = torch.from_numpy(A_norm).to(device)

print("Adjacency:", A_norm.shape)
print("Nonzero:", int(np.count_nonzero(A_norm)))

In [ ]:
# 5. Strict chronological split + TRAIN-ONLY scaler
T = len(speed)
train_end = int(T * 0.70)
val_end = int(T * 0.80)

mean = speed[:train_end].mean(axis=0).astype(np.float32)
std = np.maximum(speed[:train_end].std(axis=0), 1e-3).astype(np.float32)

speed_z = ((speed - mean) / std).astype(np.float32)

print("Train:", train_end)
print("Val  :", val_end - train_end)
print("Test :", T - val_end)
print("Scaler fitted only on:", f"[0:{train_end})")

In [ ]:
# 6. Time features
minutes = timestamps.hour.to_numpy() * 60 + timestamps.minute.to_numpy()
tod = (minutes / (24 * 60)).astype(np.float32)

time_features = np.stack([
    np.sin(2*np.pi*tod),
    np.cos(2*np.pi*tod)
], axis=1).astype(np.float32)

INPUT_STEPS = 12
HORIZON = 6
BATCH_SIZE = 64

print("Time features:", time_features.shape)

In [ ]:
# 7. Windowed datasets
class TrafficWindowDataset(Dataset):
    def __init__(self, speed_z, time_features, start, end, input_steps, horizon):
        self.speed = speed_z
        self.time = time_features
        self.start = int(start)
        self.end = int(end)
        self.input_steps = int(input_steps)
        self.horizon = int(horizon)

        # Prediction timestamps must be fully inside this split.
        self.first_t = max(self.start, self.input_steps)
        self.last_t = self.end - self.horizon

        if self.last_t <= self.first_t:
            raise ValueError("Split is too small.")

    def __len__(self):
        return self.last_t - self.first_t + 1

    def __getitem__(self, i):
        t = self.first_t + i

        # History may use observations immediately before the split boundary.
        s = self.speed[t-self.input_steps:t]           # [Tin, N]
        tf = self.time[t-self.input_steps:t]            # [Tin, 2]
        tf = np.broadcast_to(tf[:, None, :], (self.input_steps, s.shape[1], 2))

        x = np.concatenate([s[..., None], tf], axis=-1).astype(np.float32)
        y = self.speed[t:t+self.horizon].astype(np.float32)

        return torch.from_numpy(x), torch.from_numpy(y)

train_ds = TrafficWindowDataset(speed_z, time_features, 0, train_end, INPUT_STEPS, HORIZON)
val_ds   = TrafficWindowDataset(speed_z, time_features, train_end, val_end, INPUT_STEPS, HORIZON)
test_ds  = TrafficWindowDataset(speed_z, time_features, val_end, T, INPUT_STEPS, HORIZON)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                          pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                        pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                         pin_memory=True)

X0, Y0 = next(iter(train_loader))
assert X0.shape[1:] == (INPUT_STEPS, 325, 3)
assert Y0.shape[1:] == (HORIZON, 325)
print("Train/Val/Test windows:", len(train_ds), len(val_ds), len(test_ds))
print("Batch X:", X0.shape, "| Y:", Y0.shape)

In [ ]:
# 8. Enhanced STGCN
class GatedTemporalConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dropout=0.10):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels * 2,
            kernel_size=(kernel_size, 1),
            padding=(kernel_size // 2, 0)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h, g = self.conv(x).chunk(2, dim=1)
        return self.dropout(h * torch.sigmoid(g))

class GraphConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x, A):
        x = torch.einsum("ij,bctj->bcti", A, x)
        return self.proj(x)

class EnhancedSTGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.10):
        super().__init__()
        self.temporal1 = GatedTemporalConv(in_channels, out_channels, 3, dropout)
        self.graph = GraphConv(out_channels, out_channels)
        self.temporal2 = GatedTemporalConv(out_channels, out_channels, 3, dropout)
        self.residual = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        self.norm = nn.GroupNorm(1, out_channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, A):
        r = self.residual(x)
        x = F.gelu(self.temporal1(x))
        x = F.gelu(self.graph(x, A))
        x = self.temporal2(x)
        x = self.dropout(self.norm(x))
        return F.gelu(x + r)

class EnhancedSTGCN(nn.Module):
    def __init__(self, num_nodes=325, input_channels=3, hidden=64, horizon=6, dropout=0.10):
        super().__init__()
        self.input_proj = nn.Conv2d(input_channels, hidden, 1)
        self.block1 = EnhancedSTGCNBlock(hidden, hidden, dropout)
        self.block2 = EnhancedSTGCNBlock(hidden, 96, dropout)
        self.block3 = EnhancedSTGCNBlock(96, 96, dropout)
        self.block4 = EnhancedSTGCNBlock(96, hidden, dropout)
        self.head = nn.Sequential(
            nn.Conv2d(hidden, hidden, 1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv2d(hidden, horizon, 1)
        )

    def forward(self, x, A):
        x = x.permute(0, 3, 1, 2).contiguous()   # B,C,T,N
        x = self.input_proj(x)
        x = self.block1(x, A)
        x = self.block2(x, A)
        x = self.block3(x, A)
        x = self.block4(x, A)
        x = x[:, :, -1:, :]
        return self.head(x).squeeze(2)             # B,H,N

model = EnhancedSTGCN().to(device)
with torch.no_grad():
    test_out = model(X0[:2].to(device), A_tensor)
assert test_out.shape == (2, HORIZON, 325)
print("Forward-pass OK:", tuple(test_out.shape))
print("Parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# 9. Training
criterion = nn.SmoothL1Loss(beta=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-5
)

EPOCHS = 30
PATIENCE = 6
GRAD_CLIP = 3.0
MODEL_PATH = "/content/CPSTN_Enhanced_PEMS_BAY_best.pt"

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_items = 0

    for X, Y in loader:
        X = X.to(device, non_blocking=True)
        Y = Y.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        pred = model(X, A_tensor)
        loss = criterion(pred, Y)

        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        bs = X.size(0)
        total_loss += loss.item() * bs
        total_items += bs

    return total_loss / total_items

best_val = float("inf")
bad_epochs = 0
history = {"train": [], "val": [], "lr": []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = run_epoch(train_loader, True)
    val_loss = run_epoch(val_loader, False)
    scheduler.step(val_loss)

    lr = optimizer.param_groups[0]["lr"]
    history["train"].append(train_loss)
    history["val"].append(val_loss)
    history["lr"].append(lr)

    improved = val_loss < best_val
    if improved:
        best_val = val_loss
        bad_epochs = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "mean": mean,
            "std": std,
            "adjacency": A_norm,
            "input_steps": INPUT_STEPS,
            "horizon": HORIZON,
            "num_nodes": 325,
            "input_channels": 3,
            "hidden": 64,
            "dropout": 0.10,
            "model_name": "EnhancedSTGCN",
            "best_val_loss": float(best_val),
            "epoch": epoch
        }, MODEL_PATH)
    else:
        bad_epochs += 1

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train {train_loss:.5f} | Val {val_loss:.5f} | LR {lr:.2e} | {time.time()-t0:.1f}s" + (" <-- best" if improved else ""))

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

print("BEST VAL LOSS:", best_val)
print("CHECKPOINT:", MODEL_PATH)

In [ ]:
# 10. Training curves
plt.figure(figsize=(10,5))
plt.plot(history["train"], label="Train")
plt.plot(history["val"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("SmoothL1 loss")
plt.title("CPSTN Enhanced — PEMS-BAY")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
# 11. Verify the saved checkpoint can be loaded and produces the expected output
try:
    ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(MODEL_PATH, map_location=device)

required = ["model_state_dict","mean","std","adjacency","input_steps","horizon","num_nodes","input_channels","hidden"]
missing = [k for k in required if k not in ckpt]
if missing:
    raise RuntimeError("Checkpoint missing fields: " + str(missing))

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

with torch.no_grad():
    out = model(X0[:2].to(device), A_tensor)

assert out.shape == (2, HORIZON, 325)
assert np.allclose(ckpt["mean"], mean)
assert np.allclose(ckpt["std"], std)
assert ckpt["input_steps"] == INPUT_STEPS
assert ckpt["horizon"] == HORIZON

print("Checkpoint integrity: PASS")
print("NOTE: test set was not used to choose this checkpoint.")

In [ ]:
# 12. Download trained checkpoint
from google.colab import files
files.download(MODEL_PATH)